# The growth-law family — completed with von Bertalanffy

Spec §2 lists five unperturbed growth laws in scope: exponential, logistic, Gompertz, Simeoni
(exp→linear), and **von Bertalanffy / power-law**. The last was missing. v0.40 adds it: the classic
surface-area-limited model `dV/dt = a·V^(2/3) − b·V` — proliferation scales with the tumor *surface*
(nutrient/oxygen access at the rim), loss with *volume*. It is sub-exponential from the first cell and
approaches a carrying capacity `V∞ = (a/b)³`. *Unperturbed growth law; illustrative values; unverified.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.export.registry import get_kernel, kernel_values

ds = onkos.load()

## A closed form, round-trip validated

The substitution `u = V^(1/3)` linearizes the ODE, giving
`V(t) = (c + (V0^{1/3} − c)·e^{−bt/3})³` with `c = a/b = V∞^{1/3}`. Like every ODE kernel it is
round-trip validated (analytic↔SciPy integration, SBML MathML per state, NONMEM/rxode2/Pumas params).

In [ ]:
spec = get_kernel(ds['growth_laws.von_bertalanffy'])
v = kernel_values(ds['growth_laws.von_bertalanffy']); v['V0'] = 10.0
a, b = v['a'], v['b']
print(f'a={a}  b={b}   carrying capacity V_inf=(a/b)^3 = {(a/b)**3:.1f} mm')
t = np.linspace(0, 200, 401)
V = spec.analytic(t, v)
print(f'V(0)={V[0]:.1f}  V(100)={np.interp(100,t,V):.1f}  V(200)={V[-1]:.1f}')
print('stationary at V_inf (dV/dt=0):', round(spec.rhs(0.0,[(a/b)**3],v)[0], 9))

## The signature that separates the laws

The specific growth rate `(1/V)dV/dt` is the fingerprint of a growth law: exponential is flat,
logistic declines linearly in V, Gompertz declines in log V, and von Bertalanffy declines as
`a·V^{−1/3} − b` — surface-limited from the start. Same early data, very different extrapolation.

In [ ]:
laws = ['exponential','logistic','gompertz','von_bertalanffy']
print(f"{'size (mm)':>10} " + ' '.join(f'{l[:7]:>8}' for l in laws))
for V in (10, 50, 150):
    row = []
    for l in laws:
        s = get_kernel(ds[f'growth_laws.{l}']); vv = kernel_values(ds[f'growth_laws.{l}'])
        row.append(s.rhs(0.0, [float(V)], vv)[0] / V)
    print(f'{V:>10} ' + ' '.join(f'{r:>8.4f}' for r in row))

## The takeaway

The growth-law family declared in spec §2 is now complete. Von Bertalanffy fills the surface-limited /
power-law slot: a canonical, closed-form, round-trip-validated kernel whose sub-exponential signature is
distinct from the others — so it is another option in the growth-law model-selection axis, and it
predicts a slower late approach to carrying capacity than logistic or Gompertz from the same early data.